In [0]:
from pyspark.sql import functions as F


try:
    silver = spark.table("project.silver.orders_enriched")
except Exception:
    # create an empty gold table and exit
    spark.sql("""
      CREATE OR REPLACE TABLE project.gold.sales_daily
      (order_day DATE, orders BIGINT, customers BIGINT)
      USING DELTA
    """)
    print("Silver missing (transform skipped and never ran before). Created empty gold table.")
    dbutils.notebook.exit("DONE")

gold = (silver
        .groupBy(F.to_date("order_ts").alias("order_day"))
        .agg(
            F.count("*").alias("orders"),
            F.countDistinct("customer_id").alias("customers")
        )
       )

(gold.write.format("delta")
 .mode("overwrite")
 .saveAsTable("project.gold.sales_daily"))

print("Published project.gold.sales_daily")
display(gold.orderBy("order_day"))
